Transpiler avec SpinPulse

Le tutoriel officiel de SpinPulse présente exactement cette démarche : créer HardwareSpecs, appeler gate_transpile, construire un PulseCircuit, puis examiner la fidélité et l'effet du bruit.

In [ ]:
from qiskit import QuantumCircuit
from spin_pulse import PulseCircuit
from spin_pulse.transpilation.hardware_specs import (
    HardwareSpecs,
    Shape,
)

n_qubits = 3

hardware = HardwareSpecs(
    n_qubits,
    0.5,    # B_field
    0.3,    # delta
    0.03,   # J_coupling
    Shape.GAUSSIAN,
    5,      # ramp_duration
    optim=3,
)

qc = QuantumCircuit(n_qubits)
qc.h(0)
qc.cx(0, 1)
qc.cx(1, 2)

isa_circuit = hardware.gate_transpile(qc)

print("Circuit logique :")
print(qc.draw())

print("Circuit compilé :")
print(isa_circuit.draw())

pulse_circuit = PulseCircuit.from_circuit(
    isa_circuit, hardware
)

print("Fidélité du circuit à impulsions :")
print(pulse_circuit.fidelity())
print("Profondeur logique :", qc.depth())
print("Portes logiques :", qc.count_ops())

print("Profondeur compilée :", isa_circuit.depth())
print("Portes compilées :", isa_circuit.count_ops())


Circuit logique :
     ┌───┐          
q_0: ┤ H ├──■───────
     └───┘┌─┴─┐     
q_1: ─────┤ X ├──■──
          └───┘┌─┴─┐
q_2: ──────────┤ X ├
               └───┘
Circuit compilé :
global phase: π
         ┌──────────┐  ┌─────────┐  ┌─────────────┐           ┌───────┐ »
q_0 -> 0 ┤ Rz(-π/2) ├──┤ Rx(π/2) ├──┤ Rz(0.75942) ├─■─────────┤ Rx(π) ├─»
         ├──────────┤┌─┴─────────┴─┐└─────────────┘ │ZZ(-π/4) ├───────┴┐»
q_1 -> 1 ┤ Ry(-π/2) ├┤ Rz(-1.6129) ├────────────────■─────────┤ Ry(-π) ├»
         ├──────────┤└─┬─────────┬─┘ ┌────────────┐           └────────┘»
q_2 -> 2 ┤ Rz(-π/2) ├──┤ Rx(π/2) ├───┤ Rz(2.9101) ├─────────────────────»
         └──────────┘  └─────────┘   └────────────┘                     »
«                              ┌─────────────┐                                 »
«q_0 -> 0 ───────────■─────────┤ Rz(-2.3822) ├─────────────────────────────────»
«         ┌────────┐ │ZZ(-π/4) └┬────────────┤┌──────────┐           ┌────────┐»
«q_1 -> 1 ┤ Rz(-π) ├─■──────────┤ Rz(3.0

# Analyse de `HardwareSpecs` dans SpinPulse

```python
hardware = HardwareSpecs(
    n_qubits,
    0.5,   # B_field
    0.2,   # delta
    0.01,  # J_coupling
    Shape.GAUSSIAN,
    5,     # ramp_duration
    optim=3,
)
```

## 1. Que représentent les paramètres ?

Unités du modèle : $\hbar = 1$, les énergies sont des **pulsations** (rad par $t_s$) et les temps sont des multiples du pas d'échantillonnage $t_s$ (par exemple $t_s = 1$ ns, donc rad/ns).

| Paramètre | Valeur | Signification physique |
|---|---|---|
| `n_qubits` | — | Nombre de qubits de spin, alignés sur une **chaîne 1D** (couplage natif entre $i$ et $i+1$ uniquement) |
| `B_field` | 0,5 | **Valeur maximale** de l'amplitude $B_i(t)$ du champ de pilotage (ESR) : sert aux rotations $R_X$, $R_Y$ |
| `delta` | 0,2 | **Valeur maximale** du décalage de fréquence de Larmor $\delta\omega_i(t)$ (effet Stark, contrôlé en DC) : sert à $R_Z$ et au désaccord $\Delta$ des $R_{ZZ}$ |
| `J_coupling` | 0,01 | **Valeur maximale** du couplage d'échange $J_{i,j}(t)$ entre voisins : sert à $R_{ZZ}$ |
| `Shape.GAUSSIAN` | — | Forme d'enveloppe des impulsions (l'autre option est `SQUARE`) |
| `5` | 5 | Voir « Point à vérifier » ci-dessous |
| `optim=3` | 3 | Niveau d'optimisation du transpileur Qiskit (0 à 3, défaut 0) pour la première passe de transpilation |

D'après **arXiv:2601.10435**

La condition de validité du modèle de $R_{ZZ}$ est $J \ll \Delta$. Ici $J/\Delta = 0{,}05$. L'article cite, pour $J = 0{,}03$ et $\Delta = 0{,}3$ (rapport 0,1), une fidélité de processus entre circuit idéal et circuit à impulsions de 0,99999. Un $J$ plus petit améliore cette fidélité mais allonge les portes.

## 2. Modèle physique : qubits de spin en boîtes quantiques, couplage de Heisenberg


Le modèle est celui de spins ½ piégés dans des boîtes quantiques (silicium). Dans le référentiel tournant à la fréquence de Larmor :

**Un qubit** (pilotage ESR + décalage Stark) :
$$h_i(t) = \frac{B_i(t)}{2}\big(\cos\varphi_i\,X_i + \sin\varphi_i\,Y_i\big) + \frac{\delta\omega_i(t)}{2}\,Z_i$$

**Deux voisins** (échange de Heisenberg, isotrope) :
$$h_{i,j}(t) = \frac{J_{i,j}(t)}{2}\big(X_iX_j + Y_iY_j + Z_iZ_j\big)$$

Le bruit par défaut est un bruit de phase classique non markovien, $\delta h_i = \frac{\epsilon_i(t)}{2}Z_i$ (quasi-statique, blanc ou rose, via `ExperimentalEnvironment`).

## 3. Les 4 portes natives et comment elles sont réalisées

Base native : `rx`, `ry`, `rz`, `rzz`, avec $R_\sigma(\theta) = e^{-i\theta\sigma/2}$.

| Porte | Réalisation physique | Durée minimale (ordre de grandeur) |
|---|---|---|
| $R_X(\theta)$ | Impulsion de champ $B$ avec $\int B\,dt = \theta$ | $\theta / B_{\max}$ : environ 6,3 $t_s$ pour $\pi$ |
| $R_Y(\theta)$ | Idem avec phase $\varphi = \pi/2$ | Idem |
| $R_Z(\theta)$ | Pas de champ ; impulsion de décalage Stark avec $\int \delta\omega\,dt = \theta$ | $\theta / \delta_{\max}$ : environ 15,7 $t_s$ pour $\pi$ |
| $R_{ZZ}(\theta)$ | Rampe de $\Delta$, puis impulsion adiabatique de $J$ (transformation de Schrieffer-Wolff), avec $\theta = \int J\,dt$ | $\theta / J_{\max}$ : environ 157 $t_s$ pour $\pi/2$, plus les rampes |

Ce sont des **bornes inférieures** : une enveloppe gaussienne et les rampes allongent les portes. Lisez les durées exactes sur `pulse_circ.plot()`.

Conséquences pratiques :
- Avec ces valeurs, $R_{ZZ}$ est environ **deux ordres de grandeur plus lent** que $R_X$. Le **nombre de `rzz`** domine la durée du circuit, donc son exposition au bruit. C'est la quantité à minimiser.
- $R_Z$ n'est **pas virtuelle** (comme elle l'est en supraconducteur) : c'est une vraie impulsion, qui coûte du temps.
- Chaque `rzz` est découpée en deux demi-portes avec des $X\otimes X$ (écho de spin). Cela annule le bruit quasi-statique.

$$R_{ZZ}(\theta) = \begin{pmatrix} e^{-i\theta/2} & 0 & 0 & 0 \\  0 & e^{i\theta/2} & 0 & 0 \\  0 & 0 & e^{i\theta/2} & 0 \\  0 & 0 & 0 & e^{-i\theta/2} \end{pmatrix}$$

## 4. Décompositions utiles (portes natives uniquement)

Convention : ordre **du circuit**, de gauche à droite (la première porte écrite est appliquée en premier). Égalités à une **phase globale** près. $c$ = contrôle, $t$ = cible.

| Porte | Décomposition | Nb de `rzz` |
|---|---|---|
| $X, Y, Z$ | `rx(π)`, `ry(π)`, `rz(π)` | 0 |
| $S$, $S^\dagger$, $T$ | `rz(π/2)`, `rz(−π/2)`, `rz(π/4)` | 0 |
| Phase $P(\lambda)$ | `rz(λ)` | 0 |
| $\sqrt{X}$ | `rx(π/2)` | 0 |
| **Hadamard** | `rz(π)` puis `ry(π/2)` (ou `rz(π/2)`, `rx(π/2)`, `rz(π/2)`) | 0 |
| **CZ** | `rz(π/2)` sur $c$ et sur $t$, puis `rzz(−π/2)` | 1 |
| **CNOT** | `ry(−π/2)` sur $t$, CZ, puis `ry(π/2)` sur $t$ | 1 |
| **Phase contrôlée** $CP(\lambda)$ | `rz(λ/2)` sur $c$ et $t$, puis `rzz(−λ/2)` | 1 |
| **$CR_Z(\lambda)$** | `rz(λ/2)` sur $t$, puis `rzz(−λ/2)` | 1 |
| $R_{XX}(\theta)$ | `ry(−π/2)` sur les 2, `rzz(θ)`, `ry(π/2)` sur les 2 | 1 |
| $R_{YY}(\theta)$ | `rx(π/2)` sur les 2, `rzz(θ)`, `rx(−π/2)` sur les 2 | 1 |
| **SWAP** | 3 CNOT, ou $R_{XX}(-\tfrac{\pi}{2})\,R_{YY}(-\tfrac{\pi}{2})\,R_{ZZ}(-\tfrac{\pi}{2})$ | 3 |
| $C$-$U$ (porte contrôlée quelconque) | 2 CNOT + rotations à un qubit | 2 |
| **Toffoli** | 6 CNOT (décomposition standard) + portes à un qubit | 6 |

Points à retenir :
- Le noyau natif à deux qubits est **une seule porte `rzz`**. CZ, CNOT, $CP$ et $CR_Z$ n'en coûtent qu'une. Cela vaut pour la QFT (qui n'utilise que des $CP$).
- Avec la **chaîne linéaire**, toute porte entre qubits non adjacents ajoute des SWAP, soit **3 `rzz` par SWAP**. Le coût d'une porte multi-contrôlée explose vite : vérifiez toujours avec `hardware.gate_transpile(circ)`.


## 5. Point à vérifier : le rôle du `5`

Dans l'article, le dernier argument positionnel est la **durée de montée** (`ramp_duration`) des champs jusqu'à leur maximum. Mais dans le code source, `ramp_duration` (défaut 1) ne semble s'appliquer qu'aux impulsions `SQUARE`, alors que les impulsions `GAUSSIAN` utilisent un paramètre séparé `coeff_duration` (défaut 5). Si la signature de votre version place `coeff_duration` après `ramp_duration`, votre `5` pourrait être ignoré en mode gaussien. À vérifier :

```python
import inspect
print(inspect.signature(HardwareSpecs))
print(hardware)   # affiche B_field, delta, J_coupling, forme, ramp_duration, coeff_duration
```

Utilisez ensuite des arguments nommés pour lever toute ambiguïté.

In [2]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator

def same(a, b):  # égalité à une phase globale près
    return Operator(a).equiv(Operator(b))

# CNOT = ry(-π/2)_t · [rz(π/2)_c rz(π/2)_t · rzz(-π/2)] · ry(π/2)_t
ref = QuantumCircuit(2); ref.cx(0, 1)
nat = QuantumCircuit(2)
nat.ry(-np.pi/2, 1)
nat.rz(np.pi/2, 0); nat.rz(np.pi/2, 1)
nat.rzz(-np.pi/2, 0, 1)
nat.ry(np.pi/2, 1)
print("CNOT :", same(nat, ref))

# Hadamard = rz(π) puis ry(π/2)
ref = QuantumCircuit(1); ref.h(0)
nat = QuantumCircuit(1); nat.rz(np.pi, 0); nat.ry(np.pi/2, 0)
print("H :", same(nat, ref))

# Phase contrôlée CP(λ) = rz(λ/2) sur c et t, puis rzz(-λ/2)
lam = 0.7
ref = QuantumCircuit(2); ref.cp(lam, 0, 1)
nat = QuantumCircuit(2)
nat.rz(lam/2, 0); nat.rz(lam/2, 1); nat.rzz(-lam/2, 0, 1)
print("CP :", same(nat, ref))

# Comparaison avec la transpilation réelle de SpinPulse
circ = QuantumCircuit(2); circ.cx(0, 1)
isa = hardware.gate_transpile(circ)
print(isa.count_ops())   # nombre de rzz, rx, ry, rz après transpilation

CNOT : True
H : True
CP : True
OrderedDict({'rz': 5, 'rx': 3, 'rzz': 2, 'ry': 2})


## Effet du bruit


In [3]:
from spin_pulse.environment.noise import NoiseType
from spin_pulse.environment.experimental_environment import (
    ExperimentalEnvironment,
)

# 1. Configuration de l'environnement expérimental et du modèle de bruit
# On définit le matériel cible, un type de bruit en 1/f (PINK), 
env = ExperimentalEnvironment(
    hardware_specs=hardware,
    noise_type=NoiseType.PINK,  # Bruit rose (1/f)
    T2S=500,                    # Temps de cohérence T2* (en unités de temps de la simulation)
    duration=2**15,             # Durée totale de la simulation (32 768 pas)
    segment_duration=2**15,     # Durée de chaque segment pour le calcul du bruit
)

# 2. Génération du circuit d'impulsions bruité (Pulse-Level Circuit)
# On convertit le circuit ISA (Instruction Set Architecture) abstrait en un circuit 
# d'impulsions physiques en y appliquant les contraintes matérielles et l'environnement bruité.
noisy_pulse = PulseCircuit.from_circuit(
    isa_circuit,
    hardware,
    exp_env=env,
)

# 3. Évaluation de la performance du circuit sous bruit
# On calcule et affiche la fidélité moyenne du circuit par rapport au cas idéal
print("Fidélité moyenne avec bruit :")
print(noisy_pulse.mean_fidelity(env))

Fidélité moyenne avec bruit :


100%|██████████| 76/76 [00:09<00:00,  8.02it/s]

0.9404717808781806
